# Boruta Ensemble Validation

This notebook runs **every strategy** on a single ticker, generates all OR-logic ensemble combinations,
ranks by IS Sharpe, then validates with **Boruta** (shuffle OOS returns to test statistical significance).

**Workflow:**
1. Run 7 individual strategies on train data
2. Generate all 2..N ensemble combinations (OR logic)
3. Rank by IS Sharpe
4. Boruta-validate top 25 ensembles on OOS data
5. Visualize and export results

In [ ]:
# !pip install vectorbt yfinance TA-Lib numpy pandas matplotlib seaborn scipy

## Imports

In [ ]:
import yfinance as yf
import talib
import numpy as np
import pandas as pd
import vectorbt as vbt
import matplotlib.pyplot as plt
import seaborn as sns
from itertools import combinations
import warnings
import json
import os

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-darkgrid')
print('All imports loaded successfully.')

## Configuration

In [ ]:
TICKER = "QQQ"
START_DATE = "2018-01-01"
TRAIN_RATIO = 0.60
INIT_CASH = 100_000
FEES = 0.0005
SLIPPAGE = 0.0005
MIN_TRADES = 10
N_SHUFFLES = 500       # Boruta shuffle count
BORUTA_THRESHOLD = 0.80  # 80% significance

print(f"Ticker: {TICKER}")
print(f"Start Date: {START_DATE}")
print(f"Train Ratio: {TRAIN_RATIO}")
print(f"Boruta Shuffles: {N_SHUFFLES}")
print(f"Boruta Threshold: {BORUTA_THRESHOLD}")

## Download Data

In [ ]:
raw = yf.download(TICKER, start=START_DATE, auto_adjust=True)

# Handle MultiIndex columns from yfinance
if isinstance(raw.columns, pd.MultiIndex):
    raw.columns = raw.columns.get_level_values(0)

close = raw['Close'].astype(float)
high = raw['High'].astype(float)
low = raw['Low'].astype(float)
open_ = raw['Open'].astype(float)
volume = raw['Volume'].astype(float)

print(f"Downloaded {len(close)} bars for {TICKER}")
print(f"Date range: {close.index[0].date()} to {close.index[-1].date()}")
print(f"Close price range: ${close.min():.2f} - ${close.max():.2f}")

## Train / Test Split

In [ ]:
split_idx = int(len(close) * TRAIN_RATIO)

# Full data references (for signal generation on full data)
close_full = close
high_full = high
low_full = low

# Train data
close_train = close.iloc[:split_idx]
high_train = high.iloc[:split_idx]
low_train = low.iloc[:split_idx]

# Validation data
close_val = close.iloc[split_idx:]
high_val = high.iloc[split_idx:]
low_val = low.iloc[split_idx:]

print(f"Train: {close_train.index[0].date()} to {close_train.index[-1].date()} ({len(close_train)} bars)")
print(f"Val:   {close_val.index[0].date()} to {close_val.index[-1].date()} ({len(close_val)} bars)")
print(f"Split index: {split_idx}")

## Strategy Signal Generators

Each function generates signals on the **full dataset**, then we slice to train/val for backtesting.
All strategies use 1-bar execution delay.

In [ ]:
def strat_macd(close, high=None, low=None):
    """MACD Crossover (fast=12, slow=26, signal=9)"""
    macd, signal_line, _ = talib.MACD(close, fastperiod=12, slowperiod=26, signalperiod=9)
    entries_raw = (macd > signal_line) & (macd.shift(1) <= signal_line.shift(1))
    exits_raw = (macd < signal_line) & (macd.shift(1) >= signal_line.shift(1))
    entries = entries_raw.shift(1).fillna(False).astype(bool)
    exits = exits_raw.shift(1).fillna(False).astype(bool)
    return {'name': 'MACD', 'entries': entries, 'exits': exits}


def strat_rsi(close, high=None, low=None):
    """RSI Mean Reversion (period=14, oversold=30, overbought=70)"""
    rsi = talib.RSI(close, timeperiod=14)
    entries_raw = (rsi > 30) & (rsi.shift(1) <= 30)
    exits_raw = (rsi > 70) & (rsi.shift(1) <= 70)
    entries = entries_raw.shift(1).fillna(False).astype(bool)
    exits = exits_raw.shift(1).fillna(False).astype(bool)
    return {'name': 'RSI', 'entries': entries, 'exits': exits}


def strat_ema(close, high=None, low=None):
    """EMA Crossover (fast=12, slow=26, trend=50)"""
    fast_ema = talib.EMA(close, timeperiod=12)
    slow_ema = talib.EMA(close, timeperiod=26)
    trend_sma = talib.SMA(close, timeperiod=50)
    entries_raw = (fast_ema > slow_ema) & (fast_ema.shift(1) <= slow_ema.shift(1)) & (close > trend_sma)
    exits_raw = (fast_ema < slow_ema) & (fast_ema.shift(1) >= slow_ema.shift(1))
    entries = entries_raw.shift(1).fillna(False).astype(bool)
    exits = exits_raw.shift(1).fillna(False).astype(bool)
    return {'name': 'EMA', 'entries': entries, 'exits': exits}


def strat_donchian(close, high=None, low=None):
    """Donchian Breakout (entry=20, exit=10, filter=50)"""
    upper = talib.MAX(high, timeperiod=20).shift(1)
    lower = talib.MIN(low, timeperiod=10).shift(1)
    trend_filter = talib.SMA(close, timeperiod=50).shift(1)
    entries_raw = (close > upper) & (close > trend_filter)
    exits_raw = (close < lower)
    entries = entries_raw.shift(1).fillna(False).astype(bool)
    exits = exits_raw.shift(1).fillna(False).astype(bool)
    return {'name': 'Donchian', 'entries': entries, 'exits': exits}


def strat_bbands(close, high=None, low=None):
    """Bollinger Band Mean Reversion (period=20, std=2.0)"""
    upper, mid, lower_bb = talib.BBANDS(close, timeperiod=20, nbdevup=2.0, nbdevdn=2.0)
    entries_raw = (close < lower_bb) & (close.shift(1) >= lower_bb.shift(1))
    exits_raw = (close > mid) & (close.shift(1) <= mid.shift(1))
    entries = entries_raw.shift(1).fillna(False).astype(bool)
    exits = exits_raw.shift(1).fillna(False).astype(bool)
    return {'name': 'BBands', 'entries': entries, 'exits': exits}


def strat_atr_breakout(close, high=None, low=None):
    """ATR Volatility Breakout (atr=14, sma=20, mult=2.0)"""
    atr = talib.ATR(high, low, close, timeperiod=14)
    sma = talib.SMA(close, timeperiod=20)
    atr_mult = 2.0
    entries_raw = close > (sma + atr_mult * atr)
    exits_raw = close < (sma - 1.0 * atr)
    entries = entries_raw.shift(1).fillna(False).astype(bool)
    exits = exits_raw.shift(1).fillna(False).astype(bool)
    return {'name': 'ATR_Breakout', 'entries': entries, 'exits': exits}


def strat_stochastic(close, high=None, low=None):
    """Stochastic Oscillator (k=14, d=3, oversold=20, overbought=80)"""
    slowk, slowd = talib.STOCH(
        high, low, close,
        fastk_period=14, slowk_period=3, slowk_matype=0,
        slowd_period=3, slowd_matype=0
    )
    entries_raw = (slowk > slowd) & (slowk.shift(1) <= slowd.shift(1)) & (slowk < 20)
    exits_raw = (slowk < slowd) & (slowk.shift(1) >= slowd.shift(1)) & (slowk > 80)
    entries = entries_raw.shift(1).fillna(False).astype(bool)
    exits = exits_raw.shift(1).fillna(False).astype(bool)
    return {'name': 'Stochastic', 'entries': entries, 'exits': exits}


# Generate all strategy signals on full data
strategy_funcs = [strat_macd, strat_rsi, strat_ema, strat_donchian,
                  strat_bbands, strat_atr_breakout, strat_stochastic]

strategies = []
for func in strategy_funcs:
    s = func(close_full, high=high_full, low=low_full)
    strategies.append(s)
    n_entries = s['entries'].sum()
    n_exits = s['exits'].sum()
    print(f"{s['name']:20s}  entries={n_entries:4d}  exits={n_exits:4d}")

print(f"\nTotal strategies: {len(strategies)}")

## Individual Strategy Backtest (IS + OOS)

In [ ]:
def run_backtest(entries, exits, price, init_cash=INIT_CASH, fees=FEES, slippage=SLIPPAGE):
    """Run vectorbt backtest and return portfolio object."""
    pf = vbt.Portfolio.from_signals(
        price,
        entries=entries,
        exits=exits,
        init_cash=init_cash,
        fees=fees,
        slippage=slippage,
        freq='D'
    )
    return pf


def get_metrics(pf):
    """Extract key metrics from a portfolio."""
    try:
        sharpe = pf.sharpe_ratio()
    except Exception:
        sharpe = np.nan
    try:
        total_return = pf.total_return()
    except Exception:
        total_return = np.nan
    try:
        max_dd = pf.max_drawdown()
    except Exception:
        max_dd = np.nan
    try:
        n_trades = pf.trades.count()
    except Exception:
        n_trades = 0
    return {
        'sharpe': sharpe,
        'total_return': total_return,
        'max_dd': max_dd,
        'n_trades': n_trades
    }


# Run backtests on train and val for each individual strategy
individual_results = []

for s in strategies:
    # Train (IS)
    ent_train = s['entries'].iloc[:split_idx]
    ext_train = s['exits'].iloc[:split_idx]
    pf_train = run_backtest(ent_train, ext_train, close_train)
    m_train = get_metrics(pf_train)

    # Val (OOS)
    ent_val = s['entries'].iloc[split_idx:]
    ext_val = s['exits'].iloc[split_idx:]
    pf_val = run_backtest(ent_val, ext_val, close_val)
    m_val = get_metrics(pf_val)

    individual_results.append({
        'name': s['name'],
        'is_sharpe': m_train['sharpe'],
        'is_return': m_train['total_return'],
        'is_maxdd': m_train['max_dd'],
        'is_trades': m_train['n_trades'],
        'oos_sharpe': m_val['sharpe'],
        'oos_return': m_val['total_return'],
        'oos_maxdd': m_val['max_dd'],
        'oos_trades': m_val['n_trades'],
    })

# Display
df_individual = pd.DataFrame(individual_results)
print("Individual Strategy Results")
print("=" * 100)
fmt_cols = {
    'is_sharpe': '{:.3f}', 'is_return': '{:.2%}', 'is_maxdd': '{:.2%}',
    'oos_sharpe': '{:.3f}', 'oos_return': '{:.2%}', 'oos_maxdd': '{:.2%}'
}
display_df = df_individual.copy()
for col, fmt in fmt_cols.items():
    display_df[col] = df_individual[col].apply(lambda x: fmt.format(x) if pd.notna(x) else 'N/A')
print(display_df.to_string(index=False))

## Generate Ensemble Combinations (OR Logic)

We combine every possible subset (2 or more strategies) using OR logic for entries and exits,
then backtest each on the train set.

In [ ]:
strategy_names = [s['name'] for s in strategies]

all_combos = []
for r in range(2, len(strategies) + 1):
    for combo in combinations(range(len(strategies)), r):
        all_combos.append(combo)

print(f"Total ensemble combinations: {len(all_combos)}")
print(f"(from pairs to full {len(strategies)}-strategy ensemble)")

# Backtest each combination on train data
ensemble_results = []

for combo in all_combos:
    combo_name = ' + '.join([strategies[i]['name'] for i in combo])

    # OR logic: combine entries and exits
    ent_combined = strategies[combo[0]]['entries'].iloc[:split_idx].copy()
    ext_combined = strategies[combo[0]]['exits'].iloc[:split_idx].copy()
    for i in combo[1:]:
        ent_combined = ent_combined | strategies[i]['entries'].iloc[:split_idx]
        ext_combined = ext_combined | strategies[i]['exits'].iloc[:split_idx]

    # Backtest on train
    pf_train = run_backtest(ent_combined, ext_combined, close_train)
    m_train = get_metrics(pf_train)

    if m_train['n_trades'] < MIN_TRADES:
        continue

    ensemble_results.append({
        'combo': combo,
        'name': combo_name,
        'n_strategies': len(combo),
        'is_sharpe': m_train['sharpe'],
        'is_return': m_train['total_return'],
        'is_maxdd': m_train['max_dd'],
        'is_trades': m_train['n_trades'],
    })

# Sort by IS Sharpe descending
ensemble_results.sort(key=lambda x: x['is_sharpe'] if pd.notna(x['is_sharpe']) else -999, reverse=True)

print(f"\nValid ensembles (>= {MIN_TRADES} trades): {len(ensemble_results)}")
print(f"\nTop 25 Ensembles by IS Sharpe:")
print("=" * 100)
print(f"{'Rank':<5} {'Ensemble':<55} {'IS Sharpe':>10} {'IS Return':>10} {'IS MaxDD':>10} {'Trades':>7}")
print("-" * 100)
for i, er in enumerate(ensemble_results[:25]):
    sharpe_str = f"{er['is_sharpe']:.3f}" if pd.notna(er['is_sharpe']) else 'N/A'
    ret_str = f"{er['is_return']:.2%}" if pd.notna(er['is_return']) else 'N/A'
    dd_str = f"{er['is_maxdd']:.2%}" if pd.notna(er['is_maxdd']) else 'N/A'
    print(f"{i+1:<5} {er['name']:<55} {sharpe_str:>10} {ret_str:>10} {dd_str:>10} {er['is_trades']:>7}")

## Boruta Validation

For each top ensemble, we compare the real OOS Sharpe against shadow Sharpes
computed by shuffling the OOS daily returns.

In [ ]:
def boruta_validate(strategy_returns_oos, n_shuffles=500):
    """Boruta validation: shuffle OOS returns and compare real vs shadow Sharpe."""
    returns_clean = strategy_returns_oos.dropna()
    if len(returns_clean) < 20 or returns_clean.std() == 0:
        return 0.0, np.nan, np.nan

    real_sharpe = returns_clean.mean() / returns_clean.std() * np.sqrt(252)

    shadow_sharpes = []
    vals = returns_clean.values
    for _ in range(n_shuffles):
        shuffled = np.random.permutation(vals)
        s_std = np.std(shuffled)
        if s_std == 0:
            shadow_sharpes.append(0.0)
        else:
            shadow_sharpe = np.mean(shuffled) / s_std * np.sqrt(252)
            shadow_sharpes.append(shadow_sharpe)

    boruta_score = np.mean(real_sharpe > np.array(shadow_sharpes))
    return boruta_score, real_sharpe, np.median(shadow_sharpes)


# Take top 25 ensembles
top_n = min(25, len(ensemble_results))
top_ensembles = ensemble_results[:top_n]

print(f"Running Boruta validation on top {top_n} ensembles ({N_SHUFFLES} shuffles each)...")
print()

boruta_results = []

for idx, er in enumerate(top_ensembles):
    combo = er['combo']

    # OR logic on val data
    ent_val = strategies[combo[0]]['entries'].iloc[split_idx:].copy()
    ext_val = strategies[combo[0]]['exits'].iloc[split_idx:].copy()
    for i in combo[1:]:
        ent_val = ent_val | strategies[i]['entries'].iloc[split_idx:]
        ext_val = ext_val | strategies[i]['exits'].iloc[split_idx:]

    # Backtest on val
    pf_val = run_backtest(ent_val, ext_val, close_val)
    m_val = get_metrics(pf_val)

    # Get daily returns for Boruta
    daily_returns = pf_val.daily_returns()

    # Run Boruta
    boruta_score, real_sharpe, median_shadow = boruta_validate(daily_returns, n_shuffles=N_SHUFFLES)
    verdict = 'CONFIRMED' if boruta_score >= BORUTA_THRESHOLD else 'REJECTED'

    boruta_results.append({
        'rank': idx + 1,
        'name': er['name'],
        'combo': combo,
        'n_strategies': er['n_strategies'],
        'is_sharpe': er['is_sharpe'],
        'oos_sharpe': real_sharpe,
        'oos_return': m_val['total_return'],
        'oos_maxdd': m_val['max_dd'],
        'oos_trades': m_val['n_trades'],
        'boruta_score': boruta_score,
        'median_shadow': median_shadow,
        'verdict': verdict,
    })

# Print results
print(f"{'Rank':<5} {'Ensemble':<45} {'IS Sharpe':>10} {'OOS Sharpe':>11} {'Boruta':>8} {'Verdict':>12}")
print("=" * 95)
for br in boruta_results:
    is_s = f"{br['is_sharpe']:.3f}" if pd.notna(br['is_sharpe']) else 'N/A'
    oos_s = f"{br['oos_sharpe']:.3f}" if pd.notna(br['oos_sharpe']) else 'N/A'
    b_s = f"{br['boruta_score']:.1%}"
    icon = '  CONFIRMED' if br['verdict'] == 'CONFIRMED' else '  REJECTED'
    print(f"{br['rank']:<5} {br['name']:<45} {is_s:>10} {oos_s:>11} {b_s:>8} {icon:>12}")

n_confirmed = sum(1 for br in boruta_results if br['verdict'] == 'CONFIRMED')
n_rejected = sum(1 for br in boruta_results if br['verdict'] == 'REJECTED')
print(f"\nConfirmed: {n_confirmed} / {top_n}  |  Rejected: {n_rejected} / {top_n}")

## Visualization

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(16, 20))
fig.suptitle(f'Boruta Ensemble Validation - {TICKER}', fontsize=16, fontweight='bold', y=0.98)

# --- Subplot 1: IS vs OOS Sharpe bar chart ---
ax1 = axes[0]
n_bars = len(boruta_results)
x = np.arange(n_bars)
width = 0.35

is_sharpes = [br['is_sharpe'] if pd.notna(br['is_sharpe']) else 0 for br in boruta_results]
oos_sharpes = [br['oos_sharpe'] if pd.notna(br['oos_sharpe']) else 0 for br in boruta_results]
colors_is = ['#2ecc71' if br['verdict'] == 'CONFIRMED' else '#e74c3c' for br in boruta_results]
colors_oos = ['#27ae60' if br['verdict'] == 'CONFIRMED' else '#c0392b' for br in boruta_results]

ax1.bar(x - width/2, is_sharpes, width, color=colors_is, alpha=0.8, label='IS Sharpe')
ax1.bar(x + width/2, oos_sharpes, width, color=colors_oos, alpha=0.6, label='OOS Sharpe')
ax1.set_xlabel('Ensemble Rank')
ax1.set_ylabel('Sharpe Ratio')
ax1.set_title('Top 25 Ensembles: IS vs OOS Sharpe (Green=Confirmed, Red=Rejected)')
ax1.set_xticks(x)
ax1.set_xticklabels([str(br['rank']) for br in boruta_results], fontsize=8)
ax1.legend()
ax1.axhline(y=0, color='black', linewidth=0.5)

# --- Subplot 2: Strategy inclusion heatmap ---
ax2 = axes[1]
confirmed_results = [br for br in boruta_results if br['verdict'] == 'CONFIRMED']

# Count strategy appearances
inclusion_data = {}
for sname in strategy_names:
    in_top25 = sum(1 for br in boruta_results if sname in br['name'])
    in_confirmed = sum(1 for br in confirmed_results if sname in br['name'])
    inclusion_data[sname] = {'In Top 25': in_top25, 'In Confirmed': in_confirmed}

df_inclusion = pd.DataFrame(inclusion_data).T
if len(df_inclusion) > 0 and df_inclusion.values.max() > 0:
    sns.heatmap(df_inclusion, annot=True, fmt='d', cmap='YlOrRd', ax=ax2,
                linewidths=0.5, cbar_kws={'label': 'Count'})
    ax2.set_title('Strategy Inclusion Frequency in Ensembles')
    ax2.set_ylabel('Strategy')
else:
    ax2.text(0.5, 0.5, 'No confirmed ensembles to display',
             ha='center', va='center', transform=ax2.transAxes, fontsize=14)
    ax2.set_title('Strategy Inclusion Frequency in Ensembles')

# --- Subplot 3: Equity curves for top 3 confirmed ensembles + buy-and-hold ---
ax3 = axes[2]

# Buy and hold on full data
pf_bh = vbt.Portfolio.from_holding(close_full, init_cash=INIT_CASH, fees=FEES, freq='D')
equity_bh = pf_bh.value()
ax3.plot(equity_bh.index, equity_bh.values, label='Buy & Hold', color='gray', linewidth=1.5, alpha=0.7)

# Top 3 confirmed
top3_confirmed = confirmed_results[:3]
colors_eq = ['#2ecc71', '#3498db', '#9b59b6']

for i, br in enumerate(top3_confirmed):
    combo = br['combo']
    # Full-sample signals
    ent_full = strategies[combo[0]]['entries'].copy()
    ext_full = strategies[combo[0]]['exits'].copy()
    for ci in combo[1:]:
        ent_full = ent_full | strategies[ci]['entries']
        ext_full = ext_full | strategies[ci]['exits']

    pf_full = run_backtest(ent_full, ext_full, close_full)
    equity = pf_full.value()
    short_name = br['name'] if len(br['name']) < 40 else br['name'][:37] + '...'
    ax3.plot(equity.index, equity.values, label=f"#{br['rank']} {short_name}",
             color=colors_eq[i], linewidth=1.5)

# Mark train/val split
split_date = close.index[split_idx]
ax3.axvline(x=split_date, color='red', linestyle='--', alpha=0.7, label='Train/Val Split')

ax3.set_title('Equity Curves: Top 3 Confirmed Ensembles vs Buy & Hold')
ax3.set_xlabel('Date')
ax3.set_ylabel('Portfolio Value ($)')
ax3.legend(fontsize=8, loc='upper left')
ax3.set_xlim(close.index[0], close.index[-1])

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

## Summary & Recommendations

In [ ]:
print("=" * 80)
print(f"  BORUTA ENSEMBLE VALIDATION SUMMARY - {TICKER}")
print("=" * 80)

# Individual strategy analysis
print("\n--- Individual Strategy Performance ---")
for ir in individual_results:
    is_s = f"{ir['is_sharpe']:.3f}" if pd.notna(ir['is_sharpe']) else 'N/A'
    oos_s = f"{ir['oos_sharpe']:.3f}" if pd.notna(ir['oos_sharpe']) else 'N/A'
    status = 'GOOD' if (pd.notna(ir['is_sharpe']) and ir['is_sharpe'] > 0 and
                        pd.notna(ir['oos_sharpe']) and ir['oos_sharpe'] > 0) else 'WEAK'
    print(f"  {ir['name']:20s}  IS={is_s:>7s}  OOS={oos_s:>7s}  [{status}]")

# Confirmed ensembles
print(f"\n--- Boruta-Confirmed Ensembles ({n_confirmed} of {top_n}) ---")
if confirmed_results:
    for br in confirmed_results:
        is_s = f"{br['is_sharpe']:.3f}" if pd.notna(br['is_sharpe']) else 'N/A'
        oos_s = f"{br['oos_sharpe']:.3f}" if pd.notna(br['oos_sharpe']) else 'N/A'
        print(f"  Rank {br['rank']:>2d}: {br['name']}")
        print(f"           IS Sharpe={is_s}  OOS Sharpe={oos_s}  Boruta={br['boruta_score']:.1%}")
else:
    print("  No ensembles passed Boruta validation.")
    print("  Consider: adjusting parameters, adding more strategies, or using a different ticker.")

# Recommendation
print("\n--- Recommendation ---")
if confirmed_results:
    best = confirmed_results[0]
    print(f"  Best Ensemble: {best['name']}")
    print(f"  IS Sharpe:  {best['is_sharpe']:.3f}" if pd.notna(best['is_sharpe']) else "  IS Sharpe:  N/A")
    print(f"  OOS Sharpe: {best['oos_sharpe']:.3f}" if pd.notna(best['oos_sharpe']) else "  OOS Sharpe: N/A")
    print(f"  Boruta:     {best['boruta_score']:.1%}")
    print(f"  Components: {[strategies[i]['name'] for i in best['combo']]}")
else:
    print("  No recommendation - no ensembles passed validation.")

# Correlation matrix between confirmed ensemble returns
if len(confirmed_results) >= 2:
    print("\n--- Return Correlation Matrix (Confirmed Ensembles, OOS) ---")
    corr_returns = {}
    for br in confirmed_results:
        combo = br['combo']
        ent_val = strategies[combo[0]]['entries'].iloc[split_idx:].copy()
        ext_val = strategies[combo[0]]['exits'].iloc[split_idx:].copy()
        for ci in combo[1:]:
            ent_val = ent_val | strategies[ci]['entries'].iloc[split_idx:]
            ext_val = ext_val | strategies[ci]['exits'].iloc[split_idx:]
        pf_val = run_backtest(ent_val, ext_val, close_val)
        short_name = br['name'] if len(br['name']) < 25 else br['name'][:22] + '...'
        corr_returns[short_name] = pf_val.daily_returns()

    df_corr = pd.DataFrame(corr_returns)
    print(df_corr.corr().round(3).to_string())

print("\n" + "=" * 80)

## Export Results

In [ ]:
# Prepare export data
best_confirmed_name = confirmed_results[0]['name'] if confirmed_results else 'None'

export = {
    'ticker': TICKER,
    'date_run': str(pd.Timestamp.now()),
    'train_period': f"{close_train.index[0].date()} to {close_train.index[-1].date()}",
    'val_period': f"{close_val.index[0].date()} to {close_val.index[-1].date()}",
    'n_shuffles': N_SHUFFLES,
    'boruta_threshold': BORUTA_THRESHOLD,
    'individual_results': [
        {
            'name': ir['name'],
            'is_sharpe': float(ir['is_sharpe']) if pd.notna(ir['is_sharpe']) else None,
            'is_return': float(ir['is_return']) if pd.notna(ir['is_return']) else None,
            'is_maxdd': float(ir['is_maxdd']) if pd.notna(ir['is_maxdd']) else None,
            'is_trades': int(ir['is_trades']),
            'oos_sharpe': float(ir['oos_sharpe']) if pd.notna(ir['oos_sharpe']) else None,
            'oos_return': float(ir['oos_return']) if pd.notna(ir['oos_return']) else None,
            'oos_maxdd': float(ir['oos_maxdd']) if pd.notna(ir['oos_maxdd']) else None,
            'oos_trades': int(ir['oos_trades']),
        }
        for ir in individual_results
    ],
    'top_ensembles': [
        {
            'rank': br['rank'],
            'name': br['name'],
            'n_strategies': br['n_strategies'],
            'is_sharpe': float(br['is_sharpe']) if pd.notna(br['is_sharpe']) else None,
            'oos_sharpe': float(br['oos_sharpe']) if pd.notna(br['oos_sharpe']) else None,
            'boruta_score': float(br['boruta_score']),
            'verdict': br['verdict'],
        }
        for br in boruta_results
    ],
    'confirmed_ensembles': [
        {
            'rank': br['rank'],
            'name': br['name'],
            'is_sharpe': float(br['is_sharpe']) if pd.notna(br['is_sharpe']) else None,
            'oos_sharpe': float(br['oos_sharpe']) if pd.notna(br['oos_sharpe']) else None,
            'boruta_score': float(br['boruta_score']),
        }
        for br in confirmed_results
    ],
    'recommendation': best_confirmed_name
}

# Save JSON
export_dir = os.path.join(os.path.dirname(os.path.abspath('.')), 'exports')
os.makedirs(export_dir, exist_ok=True)

json_path = os.path.join(export_dir, f'boruta_ensemble_{TICKER}.json')
with open(json_path, 'w') as f:
    json.dump(export, f, indent=2)
print(f"JSON saved to: {json_path}")

# Save CSV of top ensembles
df_export = pd.DataFrame(boruta_results)
csv_path = os.path.join(export_dir, f'boruta_ensemble_{TICKER}.csv')
df_export.to_csv(csv_path, index=False)
print(f"CSV saved to:  {csv_path}")

print("\nDone!")